# Team Challenge · Sprint 03-04 - Catálogo de películas

**Dataset:**  
- [movies.csv](./data/movies.csv)
- [ratings.csv](./data/ratings.csv)
- [tags.csv](./data/tags.csv)
- [links.csv](./data/links.csv) 

**Entregables:** Repositorio de GitHub con el código fuente. Puede ser scripts de python o notebook ordenado, código reproducible, README con cómo ejecutar el proyecto.

**Control de versiones:** gestión del proyecto con **GitHub desde el primer día**. Trabajar en equipo en paralelo con **ramas**, **pull requests** y revisión entre compañeros.

**Parte 1 (Pandas):** Data analytics (Pandas)

**Parte 2 (TMDB):** una petición `GET /3/movie/{id}` por película; definid `TMDB_API_KEY` o `TMDB_READ_ACCESS_TOKEN` en el entorno (sin subir claves al repo).

**Parte 3 (Gemini):** añadir `overview_es` a `movies10` a partir de los `overview` de TMDB; definid `GEMINI_API_KEY` o usad `getpass` (sin subir claves al repo). Requiere `pip install google-genai`.

**Parte 4 (opcional):**
- **A)** catálogo ampliado (`pitch_es`, `edad_sugerida`, `temas`).
- **B)** recomendador por género.

## Trabajo en equipo y control de versiones

- Crear el **repositorio en GitHub** al inicio (día 1–2), no al final.
- Definir una rama principal (`main`) y una para desarrollo (`develop`) y ramas por tarea (p. ej. `feature/parte1-pandas`, `feature/tmdb`, `feature/gemini`).
- Integrar el trabajo con **pull requests** hacia `develop`; al menos **una PR revisada y mergeada por miembro** del equipo.
- Evitar trabajar en en `main` es la rama de "producción"
- Resolver conflictos en ramas antes del merge.
- **README:** cómo clonar el repo, instalar dependencias, configurar claves (`TMDB_*`, `GEMINI_API_KEY`) y ejecutar el notebook o scripts.
- **No subir claves** al repositorio (usar `.gitignore` para `.env` si aplica).

## Parte 1: Data analytics (Pandas)

### 1. Ingesta de datos

- Los CSV están en **`Team_Challenges/TC_01_Sprint_03_04/data/`**.
- Cargar con **Pandas** los cuatro ficheros: `movies.csv`, `ratings.csv`, `tags.csv`, `links.csv`.
- Comprobar para cada `DataFrame`: `shape`, columnas, `dtypes`, `head` y conteo de nulos.

In [124]:
# --- Apartado 1: Ingesta de datos ---
import pandas as pd
from IPython.display import display

df_movies = pd.read_csv("./data/movies.csv")
df_ratings = pd.read_csv("./data/ratings.csv")
df_tags = pd.read_csv("./data/tags.csv")
df_links = pd.read_csv("./data/links.csv")

dataframes = {
    "movies": df_movies,
    "ratings": df_ratings,
    "tags": df_tags,
    "links": df_links,
}

def comprobar_dataframe(nombre, df):
    print(f"\n{'*'*55}")
    print(f"\tDataFrame {nombre}")
    print(f"{'*'*55}")
    print(f"Shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print(f"\nDtypes: \n{df.dtypes}")
    print(f"\nHead: \n{df.head(3)}")
    print(f"\nConteo nulos: \n{df.isnull().sum()}")
    print(f"{'-'*55}")
    
    
for nombre, df in dataframes.items():
    comprobar_dataframe(nombre, df)
    


*******************************************************
	DataFrame movies
*******************************************************
Shape: (9742, 3)
Columns: ['movieId', 'title', 'genres']

Dtypes: 
movieId    int64
title        str
genres       str
dtype: object

Head: 
   movieId                    title  \
0        1               Toy Story    
1        2           Jumanji (1995)   
2        3  Grumpier Old Men (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  

Conteo nulos: 
movieId    0
title      0
genres     0
dtype: int64
-------------------------------------------------------

*******************************************************
	DataFrame ratings
*******************************************************
Shape: (100836, 4)
Columns: ['userId', 'movieId', 'rating', 'timestamp']

Dtypes: 
userId         int64
movieId        i

### 2. Columna `year` desde el título

- En el dataset de películas. el año de estreno suele aparecer **entre paréntesis al final** de `title`, p. ej. `Batman (1989)`.
- Implementad una función (`year_from_title`) que devuelva un entero de cuatro cifras o valor ausente (`NaN` / `<NA>`) si el título no sigue ese patrón.
- Añadid la columna **`year`** a `movies` como numérico (`pd.to_numeric(..., errors="coerce")`).
- Editar el campo `title` para que no contenga el año de estreno.
- Contad cuántas películas **no** tienen año reconocible y mostrad **una pequeña muestra** de sus títulos (casos límite).

In [ ]:
import re

# --- Apartado 2: columna `year` ---
def year_from_title(title:str):
    year_match = re.search(r"\((\d{4})\)", title)
    if year_match:
        year = int(year_match.group(1))
        return year
    else:
        return "NaN"
    
df_movies['year'] = pd.to_numeric(df_movies['title'].apply(year_from_title), errors="coerce").astype('Int64')

df_movies['title'] = df_movies['title'].str.replace(r'\s*\(\d{4}\)\s*$', "", regex=True)

df_no_year_movies = df_movies[df_movies['year'].isna()]
print(f'Del total de {len(df_movies)} movies encontramos {len(df_no_year_movies)} sin year reconocible en el title')

df_no_year_movies['title'].head(3)


Del total de 9742 movies encontramos 14 sin year reconocible en el title


0             Toy Story 
6059           Babylon 5
9031    Ready Player One
Name: title, dtype: str

### 2. Unificación de datos (*merge* / *join*)

- Entender las **claves** entre tablas (p. ej. `movieId` une `movies`, `ratings`, `tags` y `links`).
- Construir un esquema unificado: al menos una tabla **película–usuario–rating** (ratings enriquecida con título y géneros) y otra con **película–tags** si procede.
- Usar `merge` (u operaciones equivalentes) con criterio claro: tipo de unión (*inner* / *left*), duplicados generados y cómo se resuelven.
- Imprimir las 10 primeras filas de las tablas resultantes.
- Dejar documentado qué filas se pierden o se multiplican al unir y **por qué** es aceptable en vuestro caso de uso.

In [ ]:
# --- Apartado 3: Merge / join  ---

# película–usuario–rating
# mergeamos sobre ratings puesto que se nos piden ratings enriquecidos con titles y genres 
# usamos inner puesto que si los rating no sabemos de que pelicula son porque no esta en movies, no tiene mucho valor aportar esa info
# si hay una movie sin rating al ser inner la perderemos tambien pero esta okey porque no estamos centrado en sacar ratings enriquecidos
ratings_movie = pd.merge(df_ratings, df_movies[['movieId', 'title', 'genres', 'year']], on='movieId', how='inner')

# película–tags
# aqui nos centramos en las peliculas puesto para analizar los tags de una pelicula
# se duplicara la info de peliculas por cada tag
# asi conservamos todas las películas aunque no tengan tags (NaN en columnas de tags)
movie_tags = pd.merge(df_movies, df_tags, on='movieId', how='left')

print(ratings_movie.head(10))
print(movie_tags.head(10))

print(f"\nRatings original: {len(df_ratings)} filas")
print(f"Ratings + movies: {len(ratings_movie)} filas")
print(f"Diferencia: {len(df_ratings) - len(ratings_movie)} ratings perdidos (películas no encontradas en movies)")

print(f"\nMovies original: {len(df_movies)} filas")
print(f"Movies + tags: {len(movie_tags)} filas")
print(f"Películas sin ningún tag: {movie_tags['tag'].isna().sum()}")
print(f"Películas con al menos un tag: {movie_tags['tag'].notna().sum()}")


   userId  movieId  rating  timestamp                 title  \
0       1        1     4.0  964982703            Toy Story    
1       1        3     4.0  964981247      Grumpier Old Men   
2       1        6     4.0  964982224                  Heat   
3       1       47     5.0  964983815  Seven (a.k.a. Se7en)   
4       1       50     5.0  964982931   Usual Suspects, The   
5       1       70     3.0  964982400   From Dusk Till Dawn   
6       1      101     5.0  964980868         Bottle Rocket   
7       1      110     4.0  964982176            Braveheart   
8       1      151     5.0  964984041               Rob Roy   
9       1      157     5.0  964984100        Canadian Bacon   

                                        genres  year  
0  Adventure|Animation|Children|Comedy|Fantasy  <NA>  
1                               Comedy|Romance  1995  
2                        Action|Crime|Thriller  1995  
3                             Mystery|Thriller  1995  
4                       Crime|M

### 4. Agregaciones y segmentación

- Usar `groupby` (por usuario, por película o por género) con al menos una agregación multi-columna (`agg`).
- Comparar dos segmentos (p. ej. usuarios con muchas valoraciones vs. pocos; o por **década** usando la columna `year` del apartado 3) con tablas resumen.

In [ ]:
# --- Apartado 4: Agregaciones ---
# sacamos stats por usuario asi que agrupamos por userId y luego mo
stats_por_usuario = ratings_movie.groupby('userId').agg(
    num_ratings=('rating', 'count'),
    media_rating=('rating', 'mean'),
    min_rating=('rating', 'min'),
    max_rating=('rating', 'max')
).reset_index().sort_values('num_ratings', ascending=False)

display(stats_por_usuario)

stats_por_year = ratings_movie.dropna(subset=['year']).groupby('year').agg(
    num_peliculas=('movieId', 'nunique'),
    num_ratings=('rating', 'count'),
    media_rating=('rating', 'mean')
).reset_index().sort_values('media_rating', ascending=False)

stats_por_year['media_rating'] = stats_por_year['media_rating'].round(2)

display(stats_por_year)

# Comparacion sacamos cual seria la mediana de ratings por usuario  
mediana = stats_por_usuario['num_ratings'].median()
# muchas valoraciones -> mayor que la mediana 
# pocas valoraciones -> menor que la mediana
#  usamos mediana por si hay usuarios con muchisimas valoraciones no afecte demasiado y sea mejor el valor resultante
muchos_rating = stats_por_usuario[stats_por_usuario['num_ratings'] >= mediana]
pocos_rating = stats_por_usuario[stats_por_usuario['num_ratings'] < mediana]

comparacion = pd.DataFrame({
    'segmento': ['muchos rating', 'poco rating'],
    'num_usuarios': [len(muchos_rating), len(pocos_rating)],
    'media_ratings': [muchos_rating['num_ratings'].mean(), pocos_rating['num_ratings'].mean()],
    'media_valoracion': [muchos_rating['media_rating'].mean(), pocos_rating['media_rating'].mean()]
})

comparacion['media_ratings'] = comparacion['media_ratings'].round(2)
comparacion['media_valoracion'] = comparacion['media_valoracion'].round(2)


display(comparacion)


,userId,num_ratings,media_rating,min_rating,max_rating
413,414,2698,3.391957,0.5,5.0
598,599,2478,2.642050,0.5,5.0
473,474,2108,3.398956,0.5,5.0
447,448,1864,2.847371,0.5,5.0
273,274,1346,3.235884,0.5,5.0
...,...,...,...,...,...
441,442,20,1.275000,0.5,2.5
277,278,20,3.875000,1.5,5.0
146,147,20,3.375000,0.5,5.0
319,320,20,3.525000,0.5,4.0


,year,num_peliculas,num_ratings,media_rating
5,1917,1,1,4.50
17,1930,5,17,4.21
8,1921,1,5,4.10
21,1934,10,34,4.09
31,1944,16,92,4.04
...,...,...,...,...
83,1996,276,4509,3.34
19,1932,9,24,3.33
1,1903,1,2,2.50
3,1915,1,1,2.00


,segmento,num_usuarios,media_ratings,media_valoracion
0,muchos rating,305,292.23,3.61
1,poco rating,305,38.38,3.71


### 5. Preguntas sobre los datos (`movies`)

**Requisito:** haber creado la columna `year` en el **apartado 2**.

1. **Cuántas** películas están listadas en `movies`.
2. **Cuáles** son las **más antiguas** (menor año extraído del título).
3. **Cuántas** tienen **"Dracula"** en el título (coincidencia parcial, sin distinguir mayúsculas).
4. **Títulos más comunes**.
5. Películas con **"Exorcist"** ordenadas de la más antigua a la más moderna.
6. **Cuántas** con año **1950**.
7. **Cuántas** entre **1950 y 1959** inclusive.
8. **Año** de la película con título exacto **`Batman`** (y contraste con otras *Batman*).
9. Listado de películas que tienen como tag "sci-fi" y "adventure"
10. ¿Cuál es la tag más repetida?

In [ ]:
# --- Apartado 5: Preguntas sobre `movies` ---
movies_count = df_movies['movieId'].nunique()
print(f'1. Hay {movies_count} peliculas listadas en movies')

min_year = df_movies['year'].min()
movies_antiguas = df_movies[df_movies['year'] == min_year]
print(f'\n2. Hay {len(movies_antiguas)} peliculas del year {min_year} (year mas antiguo)')

display(movies_antiguas)
movies_dracula = int(df_movies['title'].str.contains('Dracula', case=False, na=False).sum())
print(f'3. Hay {movies_dracula} que tienen "Dracula" en el titulo')

titulo_comun = df_movies['title'].value_counts()
print('\n4. Los títulos más comunes son:')
for titulo, cantidad in titulo_comun.head(3).items():
    print(f"- {titulo}: {cantidad} películas")
    
print('\n5. Peliculas con "Exorcist" ordenadas de mas antigua a mas moderna:')
movies_exorcist = df_movies[df_movies['title'].str.contains('Exorcist', case=False, na=False)].sort_values('year')
display(movies_exorcist)

print('\n6. Peliculas con "Exorcist" ordenadas de mas antigua a mas moderna:')
movies_1950 = int((df_movies['year'] == 1950).sum())
movies_1950

print('\n7. Peliculas con "Exorcist" ordenadas de mas antigua a mas moderna:')
movies_range = int(df_movies['year'].between(1950, 1959).sum())
movies_range

print('\n8. Peliculas con "Exorcist" ordenadas de mas antigua a mas moderna:')

print('\n9. Peliculas con "Exorcist" ordenadas de mas antigua a mas moderna:')

print('\n10. Peliculas con "Exorcist" ordenadas de mas antigua a mas moderna:')
df_movies[df_movies['title'] == 'Batman']

1. Hay 9742 peliculas listadas en movies

2. Hay 1 peliculas del year 1902 (year mas antiguo)


,movieId,title,genres,year
5868,32898,"Trip to the Moon, A (Voyage dans la lune, Le)",Action|Adventure|Fantasy|Sci-Fi,1902


3. Hay 9 que tienen "Dracula" en el titulo

4. Los títulos más comunes son:
- Hamlet: 5 películas
- Misérables, Les: 4 películas
- Three Musketeers, The: 4 películas

5. Peliculas con "Exorcist" ordenadas de mas antigua a mas moderna:


,movieId,title,genres,year
1472,1997,"Exorcist, The",Horror|Mystery,1973
1473,1998,Exorcist II: The Heretic,Horror,1977
1474,1999,"Exorcist III, The",Horror,1990
5315,8815,Exorcist: The Beginning,Horror|Thriller,2004
5904,33644,Dominion: Prequel to the Exorcist,Horror|Thriller,2005
9173,148978,Blue Exorcist: The Movie,Animation|Fantasy|Horror|Mystery,2012


,movieId,title,genres,year
509,592,Batman,Action|Crime|Thriller,1989
5463,26152,Batman,Action|Adventure|Comedy,1966


## Parte 2: Petición HTTP a la API de TMDB

### Endpoint

`GET https://api.themoviedb.org/3/movie/{tmdb_id}`

Ejemplo público de la misma forma que en la documentación de TMDB: **`https://api.themoviedb.org/3/movie/100`** (el número es el `tmdb_id`; en vuestro caso usaréis el `tmdbId` de cada fila de `links`).

### Tareas

1. Construir un **`DataFrame` `movies10`** con **10 películas** del dataset original (por ejemplo las 10 primeras filas que tengan `tmdbId` tras unir `movies` con `links`).
2. Escribir una función **`fetch_movie_details(tmdb_id)`** que haga la petición anterior y devuelva al menos **`overview`** y **`homepage`** (texto vacío si vienen nulos).
3. Recorrer `movies10` y **añadir** a cada registro esas dos columnas en el propio `DataFrame`.

### Autenticación

Para poder acceder a la API de TMDB, debéis hacer uso del ACCESS TOKEN o de la API KEY que te proporcionan al registrarse. Recomendamos probar los endpoint en POSTMAN para hacer las pruebas de la llamada a la API antes de crear el script en Python. Deberíais tener en vuestro proyecto:**`TMDB_API_KEY`** (query `api_key`) o **`TMDB_READ_ACCESS_TOKEN`** (cabecera `Authorization: Bearer …`). Podéis crear un proyecto con variables de entorno o usar una celda de código usando **`getpass`** (entrada oculta, solo esa sesión del kernel). 

No guardéis claves en el notebook

In [280]:
# --- Parte 2: TMDB GET /movie/{id} → overview y homepage  ---
# Requiere: `movies` y `links` cargados. pip install requests
import pandas as pd
import requests
import os
from dotenv import load_dotenv
# para el renderizado de los DataFrane como si puera la ultima linea, que con print sale feo
from IPython.display import display

load_dotenv()

df_movies = pd.read_csv("./data/movies.csv")
df_links = pd.read_csv("./data/links.csv")

movies10 = pd.merge(df_movies, df_links, on='movieId', how='inner')
movies10 = movies10.head(10)

def fetch_movie_details(tmdb_id):
    url = f"https://api.themoviedb.org/3/movie/{tmdb_id}"
    params = {"api_key": os.getenv("TMDB_API_KEY")}
    
    resp = requests.get(url, params=params)
    
    if resp.status_code == 200:
        data = resp.json()
        return {
            "overview": data.get("overview", ""),
            "homepage": data.get("homepage", "")
        }
    else:
        return {"overview": "", "homepage": ""}
    
display(movies10)

for idx, row in movies10.iterrows():
    response = fetch_movie_details(int(row['tmdbId']))
    movies10.at[idx, 'overview'] = response['overview']
    movies10.at[idx, 'homepage'] = response['homepage']
    
display(movies10)

,movieId,title,genres,imdbId,tmdbId
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862.0
1,2,Jumanji (1995),Adventure|Children|Fantasy,113497,8844.0
2,3,Grumpier Old Men (1995),Comedy|Romance,113228,15602.0
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,114885,31357.0
4,5,Father of the Bride Part II (1995),Comedy,113041,11862.0
5,6,Heat (1995),Action|Crime|Thriller,113277,949.0
6,7,Sabrina (1995),Comedy|Romance,114319,11860.0
7,8,Tom and Huck (1995),Adventure|Children,112302,45325.0
8,9,Sudden Death (1995),Action,114576,9091.0
9,10,GoldenEye (1995),Action|Adventure|Thriller,113189,710.0


,movieId,title,genres,imdbId,tmdbId,overview,homepage
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862.0,"Led by Woody, Andy's toys live happily in his ...",http://toystory.disney.com/toy-story
1,2,Jumanji (1995),Adventure|Children|Fantasy,113497,8844.0,When siblings Judy and Peter discover an encha...,http://www.sonypictures.com/movies/jumanji/
2,3,Grumpier Old Men (1995),Comedy|Romance,113228,15602.0,A family wedding reignites the ancient feud be...,
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,114885,31357.0,"Cheated on, mistreated and stepped on, the wom...",
4,5,Father of the Bride Part II (1995),Comedy,113041,11862.0,Just when George Banks has recovered from his ...,
5,6,Heat (1995),Action|Crime|Thriller,113277,949.0,Obsessive master thief Neil McCauley leads a t...,https://www.20thcenturystudios.com/movies/heat
6,7,Sabrina (1995),Comedy|Romance,114319,11860.0,"After her return from school in Paris, a playb...",
7,8,Tom and Huck (1995),Adventure|Children,112302,45325.0,"A mischievous young boy, Tom Sawyer, witnesses...",
8,9,Sudden Death (1995),Action,114576,9091.0,When a man's daughter is suddenly taken during...,
9,10,GoldenEye (1995),Action|Adventure|Thriller,113189,710.0,When a powerful secret defense system is stole...,https://mgm.com/movies/goldeneye


## Parte 3: Sinopsis en español con Gemini

Usad el `DataFrame` **`movies10`** de la Parte 2 (columna `overview` en inglés desde TMDB).

### Tareas

1. Configurar **`GEMINI_API_KEY`** (variable de entorno o `getpass`, como en Sprint 4).
2. Crear el cliente **`google-genai`** y una función **`summarize_overview_es(overview, title="")`** que devuelva un **resumen en español de máximo 2 frases**. Si `overview` está vacío, devolver cadena vacía **sin llamar a la API**.
3. Recorrer `movies10` y añadir la columna **`overview_es`**.
4. Mostrar `title`, `overview` (recorte) y `overview_es` para **3 películas**.

### Autenticación

Clave en [Google AI Studio](https://aistudio.google.com/). Variable **`GEMINI_API_KEY`** o celda con **`getpass`**. No guardéis claves en el notebook.

### Prerrequisitos

- Parte 2 ejecutada (`movies10` con columna `overview`).
- `pip install google-genai`

In [130]:
# --- Parte 3: overview → overview_es con Gemini  ---
# Requiere: `movies10` de la Parte 2. pip install google-genai

## Parte 4 (opcional): extensiones con Gemini

**No obligatorio.** Solo si el grupo terminó las partes 1-3. Podéis elegir **A**, **B** o ambas. Son independientes.

---

### A) Catálogo ampliado en `movies10`

Partiendo de `movies10` (con `overview` y, si ya lo tenéis, `overview_es`), añadid con **una llamada JSON por película**:

- **`pitch_es`**: texto de cartelera en español (máx. 280 caracteres).
- **`edad_sugerida`**: uno de `TP`, `+7`, `+12`, `+16`, `+18`.
- **`temas`**: exactamente 3 temas en español (en el DataFrame, cadena separada por comas).

Función sugerida: **`enrich_catalog_fields(overview, title="", genres="", year=None)`**. Basad la respuesta solo en sinopsis y metadatos; no inventéis reparto ni datos externos. Si `overview` está vacío, no llaméis a la API.

Mostrad 2–3 filas con las columnas nuevas.

---

### B) Recomendador por género

Usad **`ratings_movies`** (Parte 1) y Gemini.

1. **`top_rated_by_genre(ratings_movies, genres, top_n=10, min_ratings=50)`** — filtra por género(s), nota media por `movieId`, devuelve el top N (mínimo `min_ratings` valoraciones por película).
2. **`recommend_movies(candidates, favorite_genres, n=3)`** — el modelo recomienda **solo** del catálogo candidato, con una frase de justificación en español cada una.
3. Probad con 1–2 géneros (p. ej. `["Action", "Sci-Fi"]`): mostrad candidatas y respuesta del modelo.

In [131]:
# --- Parte 4A (opcional): pitch_es, edad_sugerida, temas ---
# Requiere: `movies10` con `overview`; `client` y `MODEL` de la Parte 3.

In [132]:
# --- Parte 4B (opcional): recomendador por género (SOLUCIÓN) ---
# Requiere: `ratings_movies` (Parte 1); `client` y `MODEL` (Parte 3).